In [0]:
%sql
CREATE or replace TABLE question_data (
    question_id INT,
    text STRING,
    category STRING
);

--training data
INSERT INTO question_data (question_id, text, category) VALUES
    (1,'What is the capital of France?', 'geography'),
    (2,'Name the tallest mountain in the world', 'geography'),
    (3,'How does quantum computing work?', 'technology'),
    (4,'Best practices for cybersecurity', 'technology'),
    (5,'Explain the theory of relativity', 'science'),
    (6,'What are the planets in our solar system?', 'science'),
    (7,'Who discovered electricity?', 'science'),
    (8,'What is the largest desert in the world?', 'geography'),
    (9,'Difference between classical and quantum computing?', 'technology'),
    (10,'What is the circumference of the Earth?', 'geography');

num_affected_rows,num_inserted_rows
10,10


In [0]:
%sql
select * from question_data;

question_id,text,category
1,What is the capital of France?,geography
2,Name the tallest mountain in the world,geography
3,How does quantum computing work?,technology
4,Best practices for cybersecurity,technology
5,Explain the theory of relativity,science
6,What are the planets in our solar system?,science
7,Who discovered electricity?,science
8,What is the largest desert in the world?,geography
9,Difference between classical and quantum computing?,technology
10,What is the circumference of the Earth?,geography


In [0]:
%sql
WITH embeddings AS (
    SELECT
        category,
        CAST(
            ai_query(
                'databricks-bge-large-en',
                request => text
            ) AS ARRAY<FLOAT>
        ) AS text_embedding
    FROM question_data
),
query_embedding AS (
    SELECT
        CAST(
            ai_query(
                'databricks-bge-large-en',
                request => 'What is Google Map''s working principle?'
            ) AS ARRAY<FLOAT>
        ) AS embedding
)
SELECT
    max_by(
        category,
        vector_cosine_similarity(
            query_embedding.embedding,
            embeddings.text_embedding
        ),3
    ) AS most_similar_category
FROM embeddings
CROSS JOIN query_embedding;

most_similar_category
"List(technology, science, technology)"
